In [1]:
!pip -q install -U "transformers>=4.46.0" "peft==0.18.1" "accelerate>=0.33.0" "trl>=0.9.0" "bitsandbytes>=0.43.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.4.1 which is incompatible.
s3fs 2024.6.1 requires fsspec==2024.6.1.*, but you have fsspec 2025.10.0 which is incompatible.
torchaudio 2.10.0+cu126 requires torch==2.10.0, but you have torch 2.6.0+cpu which is incompatible.
torchvision 0.25.0+cu126 requires torch==2.10.0, but you have torch 2.6.0+cpu which is incompatible.


In [2]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.3.5 which is incompatible.
datasets 4.5.0 requires fsspec[http]<=2025.10.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.3.5 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.3 which is incompatible.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.4.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2

# Model Fine-Tuning

In [1]:
import warnings
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import os, sys, platform, subprocess, textwrap, json
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3080
VRAM (GB): 10.0


In [3]:
import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\user\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\user\anaconda3\Lib\site-packages

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\user\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\user\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\user\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



transformers: 5.2.0
peft: 0.18.1


In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [34]:
MODEL_ID = "google/gemma-3-270m-it"  # MIT licensed model on HF :contentReference[oaicite:5]{index=5}

TRAIN_JSONL = "./dataset/phi3_train.jsonl"
TEST_JSONL  = "./dataset/phi3_test.jsonl"

OUTPUT_DIR = "./gemma270_lora_out"

MAX_SEQ_LEN = 2048        # increase to 3072/4096 only if memory allows
TRAIN_BATCH = 4
EVAL_BATCH  = 4
GRAD_ACCUM  = 8

LR = 2e-4                 # typical LoRA LR; adjust later if unstable
EPOCHS = 1                # start 1 epoch; later 2-3 if needed

SEED = 42

In [36]:
data_files = {"train": TRAIN_JSONL, "test": TEST_JSONL}
ds = load_dataset("json", data_files=data_files)

# Keep the same eval set; only downsample train
ds["train"] = ds["train"].shuffle(seed=42).select(range(1000))

ds

DatasetDict({
    train: Dataset({
        features: ['messages', 'meta'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['messages', 'meta'],
        num_rows: 400
    })
})

In [37]:
HF_TOKEN = "" # need huggingface token that can access google gemma hf model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)

# Ensure pad token exists (many causal LMs use eos as pad)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(example):
    # add_generation_prompt=False for supervised finetuning (we already have assistant messages)
    txt = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": txt}

ds = ds.map(to_chat_text, remove_columns=[c for c in ds["train"].column_names if c != "messages"])
ds

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['messages', 'text'],
        num_rows: 400
    })
})

In [38]:
print(ds["train"][0]["text"][:800])

<bos><start_of_turn>user
You are an SEC filing Q&A assistant.
You MUST answer using ONLY the provided Context.
Every factual sentence MUST end with citations in square brackets using chunk_ids, e.g. [<chunk_id>].
You may cite multiple chunks like [id1, id2].
You MUST NOT invent chunk_ids.
If the Context is insufficient, say so clearly and cite any relevant evidence that explains the limitation.


Question: Summarize Aster Dynamics's revenue, net income, and gross margin for the year.

Context:
[demo-doc-01-xaji0y6d-00022] (pages 4-4)
Management discussed revenue drivers, including unit volumes and pricing. Operating expenses increased 11.6% due to R&D and sales investments. Management cautions that results may differ materially. Additional details are provided in the referenced section.

[


In [40]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16, 
)

from transformers import AutoConfig, AutoModelForCausalLM
import torch

# 1. Load config and wipe rope_scaling to avoid the ValueError
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)

# 2. Load the model using the modified config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    trust_remote_code=True,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,   # ✅ 3080: fp16
    attn_implementation="eager",
    token=HF_TOKEN,                  # ✅ IMPORTANT for gated
)

# Recommended prep for k-bit training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # Phi-3 sample uses this
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

trainable params: 1,898,496 || all params: 269,996,672 || trainable%: 0.7032


In [41]:
# Force LoRA/trainable params to fp16
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float16)

model.print_trainable_parameters()

trainable params: 1,898,496 || all params: 269,996,672 || trainable%: 0.7032


In [46]:
from trl import SFTConfig, SFTTrainer

# 1. Newest TRL uses 'dataset_text_field' but strictly inside SFTConfig
# If 'max_seq_length' still fails here, REMOVE it (it will auto-detect)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",       # MUST be here
    # max_seq_length=MAX_SEQ_LEN,    # Try commenting this out if it fails again
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    eval_strategy="steps",
    bf16=True,
    optim="paged_adamw_8bit",
    max_grad_norm=0.0,
    report_to=[],
)

# 2. Simplest Trainer call possible
trainer = SFTTrainer(
    model=model,
    args=sft_config,                 # Pass the SFTConfig here
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer,      # Use processing_class for 0.11.0+
)

trainer

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

In [47]:
import transformers, trl, peft, accelerate, bitsandbytes, torch
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

torch: 2.5.1+cu121
transformers: 5.2.0
trl: 0.28.0
peft: 0.18.1
accelerate: 1.12.0
bitsandbytes: 0.49.2


In [50]:
train_result = trainer.train()
train_result

{'loss': '2.085', 'learning_rate': '0.0001437', 'entropy': '1.826', 'num_tokens': '1.598e+05', 'mean_token_accuracy': '0.6407', 'epoch': '0.32'}
{'eval_loss': '1.544', 'eval_runtime': '144.6', 'eval_samples_per_second': '2.767', 'eval_steps_per_second': '0.692', 'eval_entropy': '1.812', 'eval_num_tokens': '1.598e+05', 'eval_mean_token_accuracy': '0.6934', 'epoch': '0.32'}
{'loss': '1.254', 'learning_rate': '8.125e-05', 'entropy': '1.41', 'num_tokens': '3.164e+05', 'mean_token_accuracy': '0.7329', 'epoch': '0.64'}
{'eval_loss': '1.004', 'eval_runtime': '154.6', 'eval_samples_per_second': '2.587', 'eval_steps_per_second': '0.647', 'eval_entropy': '1.077', 'eval_num_tokens': '3.164e+05', 'eval_mean_token_accuracy': '0.7924', 'epoch': '0.64'}
{'loss': '0.8725', 'learning_rate': '1.875e-05', 'entropy': '0.9654', 'num_tokens': '4.738e+05', 'mean_token_accuracy': '0.8151', 'epoch': '0.96'}
{'eval_loss': '0.7958', 'eval_runtime': '153.3', 'eval_samples_per_second': '2.609', 'eval_steps_per_sec

C:\Users\user\anaconda3\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-699e8b91-03eaba5261f48c6e66b58177;b7b93b29-682c-492f-8988-367b80e8d419)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-270m-it/resolve/main/config.json.
Access to model google/gemma-3-270m-it is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in google/gemma-3-270m-it.
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in google/gemma-3-270m-it - will assume that the vocabulary was not modified.
  warnings.warn(


{'train_runtime': '1155', 'train_samples_per_second': '0.866', 'train_steps_per_second': '0.028', 'train_loss': '1.364', 'entropy': '0.84', 'num_tokens': '4.937e+05', 'mean_token_accuracy': '0.8399', 'epoch': '1'}


TrainOutput(global_step=32, training_loss=1.364291362464428, metrics={'train_runtime': 1154.8058, 'train_samples_per_second': 0.866, 'train_steps_per_second': 0.028, 'total_flos': 389863026069504.0, 'train_loss': 1.364291362464428})

In [51]:
eval_metrics = trainer.evaluate()
eval_metrics

{'eval_loss': '0.79', 'eval_runtime': '154.7', 'eval_samples_per_second': '2.586', 'eval_steps_per_second': '0.646', 'eval_entropy': '0.8326', 'eval_num_tokens': '4.937e+05', 'eval_mean_token_accuracy': '0.8369', 'epoch': '1'}


{'eval_loss': 0.7900061011314392,
 'eval_runtime': 154.6876,
 'eval_samples_per_second': 2.586,
 'eval_steps_per_second': 0.646}

In [52]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

C:\Users\user\anaconda3\Lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-699e8c2d-371f42586e67074659fa7b63;9f516f3c-72e8-48fc-81ca-9fa234d7f254)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-270m-it/resolve/main/config.json.
Access to model google/gemma-3-270m-it is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in google/gemma-3-270m-it.
  warnings.warn(
C:\Users\user\anaconda3\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in google/gemma-3-270m-it - will assume that the vocabulary was not modified.
  warnings.warn(


Saved to: ./gemma270_lora_out


In [53]:
def generate(prompt_messages, max_new_tokens=200):
    prompt = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            use_cache=False
        )

    # Only decode the new tokens after the prompt
    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[0][prompt_len:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [54]:
sample = ds["test"][0]["messages"]
print("USER:\n", sample[1]["content"][:800], "\n")
print("MODEL OUTPUT:\n", generate(sample[:2]))

USER:
 Question: Summarize Everlight Robotics's revenue, net income, and gross margin for the year.

Context:
[demo-doc-05-5y2jr8kk-00043] (pages 4-4)
Management discussed revenue drivers, including unit volumes and pricing. Operating expenses increased 6.6% due to R&D and sales investments. Management cautions that results may differ materially. Additional details are provided in the referenced section.

[demo-doc-05-5y2jr8kk-00061] (pages 4-4)
Management discussed revenue drivers, including unit volumes and pricing. Operating expenses increased 24.1% due to R&D and sales investments. See discussion in the filing. This is subject to change based on market conditions.

[demo-doc-05-5y2jr8kk-00011] (pages 2-2)
Management discussed revenue drivers, including unit volumes and pricing. Operating expe 

MODEL OUTPUT:
 The revenue was $122.2 billion for the year. Net income was $102.2 billion. Gross margin was 15.7%. [demo-doc-05-5y2jr8kk-00043] [demo-doc-05-5y2jr8kk-00061]
